# Rail Handbrakes Data Analysis

## Overview
This notebook performs a comprehensive general data analysis on the `output.xlsx` file containing rail handbrakes data. The analysis includes:
- Data loading and initial inspection
- Data quality assessment (missing values, duplicates)
- Descriptive statistics
- Data distribution analysis
- Correlation analysis
- Visual exploration of key variables
- Summary insights and findings

**Dataset**: Rail handbrakes operational data  
**Date**: May 10, 2026  
**Purpose**: General exploratory data analysis to understand patterns, trends, and data quality

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("Libraries imported successfully!")

## 1. Data Loading
Load the Excel file and perform initial inspection of the dataset structure.

In [ ]:
# Load the data
file_path = 'output.xlsx'
df = pd.read_excel(file_path)

print(f"Dataset loaded successfully!")
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Display column names and data types
print("Column Information:")
print("=" * 60)
for idx, (col, dtype) in enumerate(zip(df.columns, df.dtypes), 1):
    print(f"{idx}. {col:30s} - {dtype}")

print(f"\nTotal columns: {len(df.columns)}")

## 2. Data Quality Assessment
Check for missing values, duplicates, and data integrity issues.

In [ ]:
# Missing values analysis
print("Missing Values Analysis:")
print("=" * 60)
missing_data = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage': (df.isnull().sum() / len(df) * 100).round(2)
})
missing_data = missing_data[missing_data['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)

if len(missing_data) > 0:
    print(missing_data.to_string(index=False))
else:
    print("✓ No missing values found in the dataset!")

print(f"\nTotal missing values: {df.isnull().sum().sum()}")

In [ ]:
# Check for duplicate rows
duplicates = df.duplicated().sum()
print(f"Duplicate Rows: {duplicates}")

if duplicates > 0:
    print(f"⚠ Found {duplicates} duplicate rows ({duplicates/len(df)*100:.2f}% of data)")
    print("\nDuplicate examples:")
    print(df[df.duplicated(keep=False)].head())
else:
    print("✓ No duplicate rows found!")

## 3. Descriptive Statistics
Generate comprehensive statistical summaries for numerical and categorical variables.

In [ ]:
# Numerical columns statistics
print("Numerical Variables - Descriptive Statistics:")
print("=" * 80)
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()

if len(numerical_cols) > 0:
    display(df[numerical_cols].describe().T.style.background_gradient(cmap='YlOrRd'))
else:
    print("No numerical columns found.")

In [ ]:
# Categorical columns analysis
print("Categorical Variables - Summary:")
print("=" * 80)
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

if len(categorical_cols) > 0:
    for col in categorical_cols:
        unique_count = df[col].nunique()
        print(f"\n{col}:")
        print(f"  Unique values: {unique_count}")
        if unique_count <= 20:  # Show value counts for columns with reasonable number of categories
            print(f"  Value counts:")
            print(df[col].value_counts().head(10).to_string())
        else:
            print(f"  Top 5 values:")
            print(df[col].value_counts().head(5).to_string())
else:
    print("No categorical columns found.")

## 4. Data Distribution Analysis
Visualize the distribution of numerical variables to understand data patterns.

In [ ]:
# Distribution plots for numerical variables
if len(numerical_cols) > 0:
    n_cols = min(3, len(numerical_cols))
    n_rows = (len(numerical_cols) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows))
    axes = axes.flatten() if len(numerical_cols) > 1 else [axes]
    
    for idx, col in enumerate(numerical_cols):
        ax = axes[idx]
        df[col].hist(bins=30, ax=ax, edgecolor='black', alpha=0.7)
        ax.set_title(f'Distribution of {col}', fontsize=12, fontweight='bold')
        ax.set_xlabel(col)
        ax.set_ylabel('Frequency')
        ax.grid(True, alpha=0.3)
        
        # Add mean and median lines
        mean_val = df[col].mean()
        median_val = df[col].median()
        ax.axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_val:.2f}')
        ax.axvline(median_val, color='green', linestyle='--', linewidth=2, label=f'Median: {median_val:.2f}')
        ax.legend()
    
    # Hide extra subplots
    for idx in range(len(numerical_cols), len(axes)):
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    plt.show()
else:
    print("No numerical columns to plot.")

In [ ]:
# Box plots to identify outliers
if len(numerical_cols) > 0:
    n_cols = min(3, len(numerical_cols))
    n_rows = (len(numerical_cols) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows))
    axes = axes.flatten() if len(numerical_cols) > 1 else [axes]
    
    for idx, col in enumerate(numerical_cols):
        ax = axes[idx]
        df.boxplot(column=col, ax=ax, patch_artist=True,
                   boxprops=dict(facecolor='lightblue', alpha=0.7),
                   medianprops=dict(color='red', linewidth=2))
        ax.set_title(f'Box Plot: {col}', fontsize=12, fontweight='bold')
        ax.set_ylabel(col)
        ax.grid(True, alpha=0.3)
    
    # Hide extra subplots
    for idx in range(len(numerical_cols), len(axes)):
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    plt.show()
else:
    print("No numerical columns to plot.")

## 5. Correlation Analysis
Examine relationships between numerical variables.

In [ ]:
# Correlation matrix
if len(numerical_cols) > 1:
    correlation_matrix = df[numerical_cols].corr()
    
    plt.figure(figsize=(12, 10))
    sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
                center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8})
    plt.title('Correlation Matrix - Rail Handbrakes Data', fontsize=14, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.show()
    
    # Find strong correlations
    print("\nStrong Correlations (|r| > 0.7):")
    print("=" * 60)
    strong_corr = []
    for i in range(len(correlation_matrix.columns)):
        for j in range(i+1, len(correlation_matrix.columns)):
            if abs(correlation_matrix.iloc[i, j]) > 0.7:
                strong_corr.append({
                    'Variable 1': correlation_matrix.columns[i],
                    'Variable 2': correlation_matrix.columns[j],
                    'Correlation': correlation_matrix.iloc[i, j]
                })
    
    if strong_corr:
        strong_corr_df = pd.DataFrame(strong_corr).sort_values('Correlation', ascending=False)
        print(strong_corr_df.to_string(index=False))
    else:
        print("No strong correlations found.")
else:
    print("Need at least 2 numerical columns for correlation analysis.")

## 6. Categorical Variable Visualization
Explore the distribution of categorical variables.

In [ ]:
# Bar plots for categorical variables
if len(categorical_cols) > 0:
    for col in categorical_cols[:5]:  # Limit to first 5 categorical columns
        if df[col].nunique() <= 20:  # Only plot if reasonable number of categories
            plt.figure(figsize=(12, 6))
            value_counts = df[col].value_counts().head(15)
            
            ax = value_counts.plot(kind='bar', color='steelblue', edgecolor='black', alpha=0.7)
            plt.title(f'Distribution of {col}', fontsize=14, fontweight='bold', pad=20)
            plt.xlabel(col, fontsize=12)
            plt.ylabel('Count', fontsize=12)
            plt.xticks(rotation=45, ha='right')
            plt.grid(axis='y', alpha=0.3)
            
            # Add value labels on bars
            for i, v in enumerate(value_counts):
                ax.text(i, v + max(value_counts)*0.01, str(v), ha='center', va='bottom', fontweight='bold')
            
            plt.tight_layout()
            plt.show()
else:
    print("No categorical columns to visualize.")

## 7. Outlier Detection
Identify potential outliers in numerical variables using IQR method.

In [ ]:
# Outlier detection using IQR method
if len(numerical_cols) > 0:
    print("Outlier Analysis (IQR Method):")
    print("=" * 80)
    
    outlier_summary = []
    
    for col in numerical_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)][col]
        outlier_count = len(outliers)
        outlier_pct = (outlier_count / len(df)) * 100
        
        outlier_summary.append({
            'Column': col,
            'Outlier_Count': outlier_count,
            'Outlier_Percentage': f"{outlier_pct:.2f}%",
            'Lower_Bound': f"{lower_bound:.2f}",
            'Upper_Bound': f"{upper_bound:.2f}"
        })
    
    outlier_df = pd.DataFrame(outlier_summary)
    print(outlier_df.to_string(index=False))
else:
    print("No numerical columns for outlier analysis.")

## 8. Data Summary Report
Comprehensive overview of the dataset characteristics.

In [ ]:
# Generate comprehensive summary report
print("="*80)
print("RAIL HANDBRAKES DATA - SUMMARY REPORT")
print("="*80)
print(f"\nAnalysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Dataset: output.xlsx")
print("\n" + "-"*80)
print("DATASET OVERVIEW")
print("-"*80)
print(f"Total Records: {len(df):,}")
print(f"Total Features: {len(df.columns)}")
print(f"  - Numerical: {len(numerical_cols)}")
print(f"  - Categorical: {len(categorical_cols)}")
print(f"\nMemory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("\n" + "-"*80)
print("DATA QUALITY")
print("-"*80)
print(f"Missing Values: {df.isnull().sum().sum():,} ({df.isnull().sum().sum()/(len(df)*len(df.columns))*100:.2f}%)")
print(f"Duplicate Rows: {df.duplicated().sum():,} ({df.duplicated().sum()/len(df)*100:.2f}%)")
print(f"Complete Records: {len(df.dropna()):,} ({len(df.dropna())/len(df)*100:.2f}%)")

if len(numerical_cols) > 0:
    print("\n" + "-"*80)
    print("NUMERICAL FEATURES SUMMARY")
    print("-"*80)
    for col in numerical_cols:
        print(f"\n{col}:")
        print(f"  Range: [{df[col].min():.2f}, {df[col].max():.2f}]")
        print(f"  Mean: {df[col].mean():.2f}")
        print(f"  Median: {df[col].median():.2f}")
        print(f"  Std Dev: {df[col].std():.2f}")
        print(f"  Skewness: {df[col].skew():.2f}")

if len(categorical_cols) > 0:
    print("\n" + "-"*80)
    print("CATEGORICAL FEATURES SUMMARY")
    print("-"*80)
    for col in categorical_cols:
        print(f"\n{col}:")
        print(f"  Unique Values: {df[col].nunique()}")
        print(f"  Most Common: {df[col].mode()[0] if len(df[col].mode()) > 0 else 'N/A'} ({df[col].value_counts().iloc[0] if len(df[col].value_counts()) > 0 else 0} occurrences)")

print("\n" + "="*80)
print("END OF REPORT")
print("="*80)

## 9. Key Insights and Observations

Based on the analysis above, document key findings:

### Data Quality
- Review missing value patterns and their potential impact
- Assess duplicate records and determine if they are legitimate
- Evaluate outliers and their business significance

### Distribution Patterns
- Identify skewed distributions that may need transformation
- Note any bimodal or multimodal distributions
- Assess normality of key variables

### Relationships
- Strong correlations between variables
- Potential multicollinearity issues
- Unexpected relationships requiring investigation

### Recommendations
- Data cleaning steps needed
- Feature engineering opportunities
- Further analysis directions
- Potential modeling considerations

In [ ]:
# Export summary statistics to CSV for reference
summary_stats = df.describe(include='all').T
summary_stats.to_csv('handbrakes_summary_statistics.csv')
print("✓ Summary statistics exported to 'handbrakes_summary_statistics.csv'")

# Save data info
with open('handbrakes_data_info.txt', 'w') as f:
    f.write("Rail Handbrakes Data - Information Report\n")
    f.write("="*80 + "\n\n")
    df.info(buf=f)
print("✓ Data info exported to 'handbrakes_data_info.txt'")